# PULSE — Entity Resolution Walkthrough

Inspect the fuzzy-match entity resolution output:
- Review `entity_aliases` with confidence distributions
- Surface ambiguous matches for human triage
- Write pending decisions to `processed_data/review_queues/aliases.jsonl`

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from agents.db import get_conn

con = get_conn()
print('Connected to PULSE DuckDB')

In [ ]:
# Allocator summary
allocators = con.execute("""
    SELECT canonical_name, allocator_type, geography, check_size_bucket, em_appetite, source_file
    FROM allocators
    ORDER BY canonical_name
""").fetchdf()

print(f'Total allocators: {len(allocators)}')
display(allocators.head(20))

In [ ]:
# Alias confidence distribution
aliases = con.execute("""
    SELECT alias_text, canonical_id, source_file, confidence, resolver_method
    FROM entity_aliases
    ORDER BY confidence ASC
""").fetchdf()

print(f'Total aliases: {len(aliases)}')

if len(aliases) > 0:
    fig, ax = plt.subplots(figsize=(8, 4))
    aliases['confidence'].hist(bins=20, ax=ax)
    ax.axvline(0.90, color='green', linestyle='--', label='Auto-match threshold (0.90)')
    ax.axvline(0.70, color='orange', linestyle='--', label='Review threshold (0.70)')
    ax.set_xlabel('Confidence')
    ax.set_title('Alias Match Confidence Distribution')
    ax.legend()
    plt.tight_layout()
    plt.show()

display(aliases.sort_values('confidence').head(20))

In [ ]:
# Review queue for aliases
from agents.reviews.queue_writer import read_queue

queued = read_queue('aliases')
print(f'Pending alias review items: {len(queued)}')

if queued:
    df_queue = pd.DataFrame(queued)
    display(df_queue[['entity_id', 'confidence', 'reason', 'surfaced_at']].head(20))

In [ ]:
# Cross-file match exploration
# Which allocators appear in more than one source file?
multi_source = con.execute("""
    SELECT canonical_id, COUNT(DISTINCT source_file) as source_count, 
           array_agg(DISTINCT source_file) as sources,
           COUNT(*) as alias_count,
           AVG(confidence) as avg_confidence
    FROM entity_aliases
    WHERE entity_type = 'allocator'
    GROUP BY canonical_id
    HAVING COUNT(DISTINCT source_file) > 1
    ORDER BY source_count DESC
""").fetchdf()

print(f'Allocators appearing in multiple sources: {len(multi_source)}')
display(multi_source)